# Проект: Многоцелевая модель для NER + event-CLS

Этот Jupyter-ноутбук - пошаговый шаблон для выполнения проекта по объединённой (multi-task) модели, решающей **NER (BIO, token-level)** и **CLS (document-level multihot событий/отношений)** на датасете NEREL.

Модель должна решать параллельно две задачи: классифицировать токены на BIO-теги и классифицировать текст на принадлежность к 30 классам-характеристикам текста.

Характеристики:
```
['WORKPLACE', 'ALTERNATIVE_NAME', 'WORKS_AS', 'PARTICIPANT_IN', 'POINT_IN_TIME', 'TAKES_PLACE_IN', 'HEADQUARTERED_IN', 'ORIGINS_FROM', 'LOCATED_IN', 'AGENT', 'AGE_IS', 'HAS_CAUSE', 'PRODUCES', 'AWARDED_WITH', 'PART_OF', 'IDEOLOGY_OF', 'MEMBER_OF', 'CONVICTED_OF', 'INANIMATE_INVOLVED', 'SUBEVENT_OF', 'SUBORDINATE_OF', 'KNOWS', 'MEDICAL_CONDITION', 'PARENT_OF', 'PLACE_RESIDES_IN', 'OWNER_OF', 'ABBREVIATION', 'FOUNDED_BY', 'ORGANIZES', 'PENALIZED_AS']
```

Внимание: вам нужно реализовать весь рабочий код - в ноутбуке реализована только загрузка датасета. Все остальные ячейки служат как инструкции / места для вашего кода.


#### Структура ноутбука

1. Подготовка окружения (пути, seed, imports)
2. EDA - загрузка данных, обзор, графики, выводы
3. Токенизация, выравнивание меток, DataLoader - реализовать `tokenize_and_align_labels`, Dataset/Collator
4. Модель (JointModel) и кастомный loss (uncertainty-weighting) - реализовать модельный класс и loss
5. Тренировка/валидация - training loop, оптимизатор, scheduler, логирование метрик
6. Инференс и анализ ошибок - реализовать inference pipeline и примеры



##### 1. EDA

Цели:

- Прочитать датасет.
- Посчитать встречаемость каждого класса в cls_vec.
- Построить графики: распределение длины текстов, число сущностей на документ.
- Написать 2–3 коротких вывода в Markdown: что можно ожидать при моделировании (редкие типы, длинные документы и т. п.).




In [36]:
from datasets import load_dataset

ds = load_dataset("danasone/nerel", split="train")
print(f"Загружено документов: {len(ds)}")

Загружено документов: 746


Task 0

Напиши код, который из переменной ds сформирует polars датафрейм со следюущими колонками:
1. id - номер текста, Проверка - уникальный в источнике 
1. size - длина текста в токенах,
1. tags - словарь-счетчик метки каждого типа в тексте, 
1. labels - словарь multihot с метками текста. Проверка - длина всегда = 30, элемент - int 0/1

Обязательные проверки целостности источника:
len(tokens) == len(tags)
Длина получившегося датафрейма = длина исходной коллекции

Информационные метрики:
кол-во документов с всеми нулями в labels
Максимальная длина документа
Медианная длина документа



In [37]:
from collections import Counter
from collections.abc import Iterable
from typing import TypedDict, cast

import polars as pl


class BaseNerelExample(TypedDict):
    tokens: list[str]
    tags: list[str]
    cls_vec: list[int]


# load_dataset со split="train" и стандартным форматом выдаёт такие записи.
examples = cast(Iterable[BaseNerelExample], ds)

# Порядок характеристик соответствует cls_vec в условии задания.
CLS_LABELS = [
    "WORKPLACE",
    "ALTERNATIVE_NAME",
    "WORKS_AS",
    "PARTICIPANT_IN",
    "POINT_IN_TIME",
    "TAKES_PLACE_IN",
    "HEADQUARTERED_IN",
    "ORIGINS_FROM",
    "LOCATED_IN",
    "AGENT",
    "AGE_IS",
    "HAS_CAUSE",
    "PRODUCES",
    "AWARDED_WITH",
    "PART_OF",
    "IDEOLOGY_OF",
    "MEMBER_OF",
    "CONVICTED_OF",
    "INANIMATE_INVOLVED",
    "SUBEVENT_OF",
    "SUBORDINATE_OF",
    "KNOWS",
    "MEDICAL_CONDITION",
    "PARENT_OF",
    "PLACE_RESIDES_IN",
    "OWNER_OF",
    "ABBREVIATION",
    "FOUNDED_BY",
    "ORGANIZES",
    "PENALIZED_AS",
]

# size — число исходных слов в tokens, до субтокенизации RuBERT.
# tags — частоты BIO-меток (B- и I- учитываются отдельно, O сохраняется).
# Словари представлены нативными Struct-колонками Polars.
tag_names = sorted({tag for example in examples for tag in example["tags"]})
# Один порядок меток для всего ds; сохраняем его и для будущего инференса.
label2id = {tag: index for index, tag in enumerate(tag_names)}
id2label = {index: tag for tag, index in label2id.items()}

# В источнике нет поля id: номер записи служит уникальным id документа.
rows = []
for doc_id, example in enumerate(examples):
    if len(example["tokens"]) != len(example["tags"]):
        raise ValueError(f"Документ {doc_id}: длины tokens и tags различаются")
    cls_vec = example["cls_vec"]
    if len(cls_vec) != len(CLS_LABELS) or any(
        type(value) is not int or value not in (0, 1) for value in cls_vec
    ):
        raise ValueError(
            f"Документ {doc_id}: ожидается cls_vec из 30 целых значений 0/1"
        )
    counts = Counter(example["tags"])
    rows.append(
        {
            "id": doc_id,
            "size": len(example["tokens"]),
            "tags": {tag: counts[tag] for tag in tag_names},
            "labels": dict(zip(CLS_LABELS, map(int, cls_vec), strict=True)),
        }
    )

eda_df = pl.DataFrame(
    rows,
    schema={
        "id": pl.Int64,
        "size": pl.Int64,
        "tags": pl.Struct({tag: pl.Int64 for tag in tag_names}),
        "labels": pl.Struct({label: pl.Int64 for label in CLS_LABELS}),
    },
)
if eda_df.height != len(ds):
    raise ValueError("Число строк eda_df не совпадает с числом документов в ds")
if eda_df["id"].n_unique() != eda_df.height:
    raise ValueError("Идентификаторы документов в eda_df не уникальны")

# Информационные метрики: длины измеряются в исходных токенах.
documents_without_labels = eda_df.filter(
    pl.all_horizontal(
        [pl.col("labels").struct.field(label) == 0 for label in CLS_LABELS]
    )
).height
max_document_size = eda_df["size"].max()
median_document_size = eda_df["size"].median()

print(f"Документов со всеми нулями в labels: {documents_without_labels}")
print(f"Максимальная длина документа в токенах: {max_document_size}")
print(f"Медианная длина документа в токенах: {median_document_size}")

eda_df.head()

Документов со всеми нулями в labels: 1
Максимальная длина документа в токенах: 1710
Медианная длина документа в токенах: 201.5


id,size,tags,labels
i64,i64,struct[58],struct[30]
0,371,"{1,0,5,6,0,3,0,0,2,1,0,0,0,0,0,0,5,25,0,19,0,0,0,3,6,0,0,2,0,0,0,1,0,0,3,0,0,2,0,0,0,0,0,0,4,7,0,20,0,0,0,0,6,0,0,1,0,249}","{1,1,0,0,0,1,1,1,1,0,1,0,0,0,1,0,0,0,0,1,0,0,0,0,0,1,0,0,1,0}"
1,503,"{1,0,2,0,0,8,0,0,9,0,0,1,0,0,0,0,1,0,0,19,0,0,29,0,22,0,2,0,0,0,0,0,0,0,7,0,0,5,0,0,0,0,0,0,0,0,0,12,0,0,18,0,39,0,0,0,0,328}","{1,1,1,1,1,1,1,1,0,1,1,1,0,0,1,1,1,0,0,0,1,0,0,0,1,0,0,0,1,0}"
2,708,"{1,0,14,23,0,15,0,0,5,2,0,0,1,0,0,0,3,5,0,26,0,0,20,0,27,0,0,0,0,3,0,0,3,0,16,0,0,1,2,0,0,0,0,0,1,0,0,9,0,0,10,0,32,0,0,0,0,489}","{1,1,1,0,1,1,1,1,1,0,1,0,0,0,1,0,1,0,0,0,1,1,0,0,0,1,1,0,1,0}"
3,331,"{1,0,7,24,0,4,0,0,2,0,0,0,1,0,0,0,1,1,0,18,0,0,5,1,14,0,0,0,0,3,0,0,3,0,8,0,0,0,0,0,0,0,0,0,0,0,0,8,0,0,4,0,27,0,0,0,0,199}","{1,1,1,1,1,1,1,1,1,0,1,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,1,0,1,0}"
4,175,"{1,0,0,0,0,7,0,0,3,0,0,0,0,0,1,0,0,3,0,14,0,1,10,0,8,0,0,0,0,0,0,0,0,0,13,0,0,1,0,0,0,0,0,0,0,1,0,7,0,0,7,0,9,0,0,0,0,89}","{1,1,1,1,1,0,1,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0}"


In [38]:
# Task 1
# build facts table
# tag: str,
# id:int,
# count: int

# Допущение - каждую BIO метку анализируем независимоƒ


# calculate metrics table:
# tag: str
# documents: int - number of unique documents with this tag
# documents_share - % share of documents with this tag among all documents
# tokens: int - number of tokens with this tag
# tokens_share - % share of all tokens in dataset
# cumulative_share - %, кумулятивная метки среди токенов, доли суммировать от самой большой до самой маленькой

# token 'O' is ignored, out of scope in the data - in text tokens and tags in this analysis

# additional total metrics:
# total documents: int
# total tokens: int

# Проверки целостности:
# 1. sum(metrics_table.count) = кол-во не O токенов в исходном датасете
# 2. max(metrics_table.documents) <= кол-во документов в датасете
# 3. unique(facts_table.id) = кол-во документов в датасете
# 4. sum(metrics_table.tokens_share) = 100%
# 5. sum(facts_table.count) = кол-во не O токенов в исходном датасете
#
# Ограничения: не должно быть проверок, не относящихся к этим пунктам. Каждая проверка снабжается комментарием-номером проверки целостности в задании


# visualization - html report
# токены в диаграммах нумеровать по порядку появления на диаграмме
# накопленную долю токенов вывести на диаграмме токенов
# диаграмму токенов по частоте документов отсортировать от самых частотных меток до самых редких сверху вниз

from math import isclose

# Считаем исходные слова, не субтокены; B- и I- учитываются отдельно.
# O исключаем, а факты оставляем только для присутствующих в документе меток.
tag_facts = (
    eda_df.select("id", "tags")
    .unnest("tags")
    .unpivot(index="id", variable_name="tag", value_name="count")
    .filter((pl.col("tag") != "O") & (pl.col("count") > 0))
    .select("tag", "id", "count")
    .sort(["tag", "id"])
)

# Итоги считаем независимо от таблицы фактов, по исходному датасету.
total_documents = len(ds)
total_tokens = sum(tag != "O" for example in examples for tag in example["tags"])

tag_metrics = (
    tag_facts.group_by("tag")
    .agg(
        pl.col("id").n_unique().alias("documents"),
        pl.col("count").sum().alias("tokens"),
    )
    .with_columns(
        (100.0 * pl.col("documents") / total_documents).alias("documents_share"),
        (100.0 * pl.col("tokens") / total_tokens).alias("tokens_share"),
    )
    .select("tag", "documents", "documents_share", "tokens", "tokens_share")
    .sort(["tokens", "tag"], descending=[True, False])
    .with_columns(pl.col("tokens_share").cum_sum().alias("cumulative_share"))
)

# В метриках агрегированный count называется tokens.
assert tag_metrics["tokens"].sum() == total_tokens
max_documents = tag_metrics.select(pl.col("documents").max()).item()
assert max_documents <= total_documents
assert tag_facts["id"].n_unique() == total_documents
assert isclose(tag_metrics["tokens_share"].sum(), 100.0, abs_tol=1e-9)
assert tag_facts["count"].sum() == total_tokens

In [40]:
from html import escape
from pathlib import Path

from IPython.display import HTML, display

metric_rows = tag_metrics.to_dicts()


def metric_bars(column, count_column, *, show_cumulative=False):
    # Каждую диаграмму сортируем по её частоте; нумерация начинается с 1.
    rows = tag_metrics.sort([count_column, "tag"], descending=[True, False]).to_dicts()
    bars = []
    for rank, row in enumerate(rows, start=1):
        cumulative_bar = ""
        cumulative_value = ""
        if show_cumulative:
            cumulative_bar = (
                '<div class="nerel-track nerel-cumulative-track">'
                f'<div style="width:{row["cumulative_share"]:.6f}%"></div></div>'
            )
            cumulative_value = (
                '<br><span class="nerel-cumulative-value">'
                f"Σ {row['cumulative_share']:.3f}%</span>"
            )
        bars.append(
            f'<div class="nerel-bar-row"><span>{rank}. {escape(row["tag"])}</span>'
            '<div class="nerel-tracks"><div class="nerel-track">'
            f'<div style="width:{row[column]:.6f}%"></div></div>{cumulative_bar}</div>'
            f"<span>{row[column]:.3f}% · {row[count_column]:,}{cumulative_value}</span></div>"
        )
    return "".join(bars)


metric_table_rows = "".join(
    f'<tr><th scope="row">{escape(row["tag"])}</th>'
    f"<td>{row['documents']:,}</td><td>{row['documents_share']:.3f}%</td>"
    f"<td>{row['tokens']:,}</td><td>{row['tokens_share']:.3f}%</td>"
    f"<td>{row['cumulative_share']:.3f}%</td></tr>"
    for row in metric_rows
)
fact_table_rows = "".join(
    f"<tr><td>{escape(row['tag'])}</td><td>{row['id']}</td><td>{row['count']}</td></tr>"
    for row in tag_facts.head(12).to_dicts()
)

report_fragment = f"""
<style>
.nerel-report {{font:14px/1.5 system-ui,sans-serif;color:#172033;background:#f5f7fb;padding:24px;border-radius:12px;max-width:1400px;margin:auto}}
.nerel-report * {{box-sizing:border-box}}
.nerel-report h1,.nerel-report h2 {{color:#172033;margin:0 0 16px}}
.nerel-report h2 {{font-size:19px}}
.nerel-report .nerel-cards {{display:grid;grid-template-columns:repeat(auto-fit,minmax(180px,1fr));gap:12px;margin:20px 0}}
.nerel-card,.nerel-panel {{background:white;border:1px solid #dce3ef;border-radius:10px;padding:18px}}
.nerel-card strong {{display:block;font-size:27px;color:#2256b8}}
.nerel-charts {{display:grid;grid-template-columns:repeat(auto-fit,minmax(340px,1fr));gap:16px;margin-bottom:20px}}
.nerel-bar-row {{display:grid;grid-template-columns:185px minmax(40px,1fr) 145px;align-items:center;gap:8px;font-size:11px;margin:5px 0}}
.nerel-bar-row>span:last-child {{text-align:right;font-variant-numeric:tabular-nums}}
.nerel-track {{height:12px;background:#edf1f8;border-radius:3px;overflow:hidden}}
.nerel-track>div {{height:100%;background:#3873da;min-width:1px}}
.nerel-track+.nerel-track {{margin-top:4px}}
.nerel-cumulative-track>div {{background:#d97706}}
.nerel-cumulative-value {{color:#a65308}}
.nerel-scroll {{overflow:auto;max-height:650px}}
.nerel-report table {{border-collapse:collapse;width:100%;background:white;font-variant-numeric:tabular-nums}}
.nerel-report th,.nerel-report td {{padding:8px 12px;border-bottom:1px solid #e5eaf2;text-align:right;white-space:nowrap}}
.nerel-report th:first-child,.nerel-report td:first-child {{text-align:left}}
.nerel-report thead th {{position:sticky;top:0;background:#eaf0fa;color:#172033}}
.nerel-report tbody tr:nth-child(even) {{background:#f7f9fc}}
.nerel-note {{color:#475569}}
.nerel-report details {{margin-top:20px}}
</style>
<article class="nerel-report" lang="ru">
<h1>NEREL — частоты BIO-меток</h1>
<p class="nerel-note">Исходные слова до субтокенизации. B- и I- учитываются отдельно; O исключена из частот и общего числа токенов.</p>
<div class="nerel-cards">
<div class="nerel-card">Документов<strong>{total_documents:,}</strong></div>
<div class="nerel-card">Исходных токенов без O<strong>{total_tokens:,}</strong></div>
<div class="nerel-card">BIO-меток без O<strong>{tag_metrics.height}</strong></div>
<div class="nerel-card">Пар документ–метка<strong>{tag_facts.height:,}</strong></div>
</div>
<div class="nerel-charts">
<section class="nerel-panel"><h2>Доля токенов, % · число токенов</h2>
<p class="nerel-note">Синий — доля метки, оранжевый (Σ) — накопленная доля. Обе шкалы — 0–100%. Метки упорядочены по убыванию числа токенов.</p>
{metric_bars("tokens_share", "tokens", show_cumulative=True)}</section>
<section class="nerel-panel"><h2>Документы с меткой, % · число документов</h2>
<p class="nerel-note">Метки упорядочены по убыванию числа документов с меткой.</p>
{metric_bars("documents_share", "documents")}</section>
</div>
<section class="nerel-panel"><h2>Метрики по всем меткам</h2>
<p class="nerel-note">Доли документов могут в сумме превышать 100%: один документ содержит несколько меток.
Сумма долей токенов: {tag_metrics["tokens_share"].sum():.2f}%. Порядок — по убыванию числа токенов. Накопленная доля — сумма долей токенов от первой строки до текущей.</p>
<div class="nerel-scroll"><table><thead><tr><th>Метка</th><th>Документов</th><th>Доля документов</th><th>Токенов</th><th>Доля токенов</th><th>Накопленная доля токенов</th></tr></thead>
<tbody>{metric_table_rows}</tbody></table></div></section>
<details><summary>Таблица фактов: первые 12 строк из {tag_facts.height:,}</summary>
<p>Одна строка — метка в документе, count &gt; 0. Полная таблица доступна в tag_facts.</p>
<table><thead><tr><th>tag</th><th>id</th><th>count</th></tr></thead><tbody>{fact_table_rows}</tbody></table></details>
</article>
"""
report_html = (
    '<!doctype html><html lang="ru"><head><meta charset="utf-8">'
    '<meta name="viewport" content="width=device-width, initial-scale=1">'
    "<title>NEREL — EDA BIO-меток</title></head><body>"
    + report_fragment
    + "</body></html>"
)
report_path = Path("reports/nerel_task1.html")
report_path.parent.mkdir(parents=True, exist_ok=True)
report_path.write_text(report_html, encoding="utf-8")
display(HTML(report_fragment))
print(f"HTML-отчёт сохранён: {report_path.resolve()}")

Метка,Документов,Доля документов,Токенов,Доля токенов,Накопленная доля токенов
B-PERSON,727,97.453%,"7,343",10.891%,10.891%
I-DATE,714,95.710%,"7,291",10.814%,21.704%
B-PROFESSION,716,95.979%,"5,193",7.702%,29.406%
I-PROFESSION,610,81.769%,"4,941",7.328%,36.735%
I-PERSON,725,97.185%,"4,871",7.224%,43.959%
B-EVENT,713,95.576%,"4,318",6.404%,50.363%
B-ORGANIZATION,666,89.276%,"3,768",5.589%,55.952%
B-DATE,723,96.917%,"3,548",5.262%,61.214%
I-ORGANIZATION,590,79.088%,"3,484",5.167%,66.381%
I-EVENT,589,78.954%,"3,347",4.964%,71.346%


HTML-отчёт сохранён: /Users/dima/Practikum/nlp-2026/sprint_1/src/project/reports/nerel_task1.html


# Task 2 EDA labels

# build label facts:
id: int - document id
label: str - text label of document
value: 1 - value of label

# build metrics:
label: str
documents: int - unique documents with this label in dataset
documents_share - % of documents with this label in dataset

# build totals - total documents

Проверки целостности:
1. max(metrics.documents) <= кол-во документов в исходном датасете
2. unique(facts.id) <= кол-во документов в исходном датасете
3. unique(label) == 30 (как в CLS_LABELS)

# Ограничения:
не должно быть проверок, не относящихся к этим пунктам. Каждая проверка снабжается комментарием-номером проверки целостности в задании

# output - html
строчки/столбцы на диаграммах нумеровать по порядку


In [41]:
# Task 2 EDA labels
from html import escape
from pathlib import Path

from IPython.display import HTML, display

# Факт — активная характеристика документа; нулевые значения исключаем.
label_facts = (
    eda_df.select("id", "labels")
    .unnest("labels")
    .unpivot(index="id", variable_name="label", value_name="value")
    .filter(pl.col("value") == 1)
    .select("id", "label", "value")
    .sort(["id", "label"])
)

label_total_documents = len(ds)
label_metrics = (
    pl.DataFrame({"label": CLS_LABELS})
    .join(
        label_facts.group_by("label").agg(pl.col("id").n_unique().alias("documents")),
        on="label",
        how="left",
    )
    .with_columns(pl.col("documents").fill_null(0))
    .with_columns(
        (100.0 * pl.col("documents") / label_total_documents).alias("documents_share")
    )
    .select("label", "documents", "documents_share")
    .sort(["documents", "label"], descending=[True, False])
)

# Проверка целостности 1: число документов с меткой не превышает размер источника.
label_max_documents = label_metrics.select(pl.col("documents").max()).item()
assert label_max_documents <= label_total_documents
# Проверка целостности 2: число уникальных документов в фактах не превышает размер источника.
assert label_facts["id"].n_unique() <= label_total_documents, (
    "Проверка 2: число уникальных документов в фактах превышает размер источника"
)
# Проверка целостности 3: в метриках ровно 30 характеристик, как в CLS_LABELS.
assert label_metrics["label"].n_unique() == len(CLS_LABELS) == 30

label_metric_rows = label_metrics.to_dicts()
label_bar_rows = "".join(
    f'<div class="cls-bar-row"><span>{rank}. {escape(row["label"])}</span>'
    f'<div class="cls-track"><div style="width:{row["documents_share"]:.6f}%"></div></div>'
    f"<span>{row['documents_share']:.2f}% · {row['documents']:,}</span></div>"
    for rank, row in enumerate(label_metric_rows, start=1)
)
label_table_rows = "".join(
    f'<tr><th scope="row">{escape(row["label"])}</th>'
    f"<td>{row['documents']:,}</td><td>{row['documents_share']:.2f}%</td></tr>"
    for row in label_metric_rows
)
label_fact_rows = "".join(
    f"<tr><td>{row['id']}</td><td>{escape(row['label'])}</td><td>{row['value']}</td></tr>"
    for row in label_facts.head(12).to_dicts()
)

label_report_fragment = f"""
<style>
.cls-report {{font:14px/1.5 system-ui,sans-serif;color:#172033;background:#f5f7fb;padding:24px;border-radius:12px;max-width:1200px;margin:auto}}
.cls-report * {{box-sizing:border-box}}
.cls-report h1,.cls-report h2 {{margin:0 0 16px}}
.cls-report h2 {{font-size:19px}}
.cls-cards {{display:grid;grid-template-columns:repeat(auto-fit,minmax(180px,1fr));gap:12px;margin:20px 0}}
.cls-card,.cls-panel {{background:white;border:1px solid #dce3ef;border-radius:10px;padding:18px}}
.cls-card strong {{display:block;font-size:27px;color:#2256b8}}
.cls-columns {{display:grid;grid-template-columns:repeat(auto-fit,minmax(min(100%,440px),1fr));gap:16px}}
.cls-bar-row {{display:grid;grid-template-columns:165px minmax(24px,1fr) 100px;align-items:center;gap:8px;font-size:11px;margin:8px 0}}
.cls-bar-row>span:last-child {{text-align:right;font-variant-numeric:tabular-nums}}
.cls-track {{height:12px;background:#edf1f8;border-radius:3px;overflow:hidden}}
.cls-track>div {{height:100%;background:#3873da}}
.cls-scroll {{overflow:auto}}
.cls-report table {{border-collapse:collapse;width:100%;font-variant-numeric:tabular-nums}}
.cls-report th,.cls-report td {{padding:8px 10px;border-bottom:1px solid #e5eaf2;text-align:right;white-space:nowrap}}
.cls-report th:first-child,.cls-report td:first-child {{text-align:left}}
.cls-report thead th {{background:#eaf0fa}}
.cls-report tbody tr:nth-child(even) {{background:#f7f9fc}}
.cls-note {{color:#475569}}
.cls-report details {{margin-top:20px}}
@media (max-width:480px) {{.cls-report {{padding:12px}} .cls-panel {{padding:10px}} .cls-bar-row {{grid-template-columns:150px minmax(16px,1fr) 85px;gap:4px;font-size:10px}}}}
</style>
<article class="cls-report" lang="ru">
<h1>NEREL — характеристики документов (CLS)</h1>
<p class="cls-note">Многометочная классификация: у документа может быть несколько характеристик.
Одна строка фактов — пара id–label со значением value = 1.</p>
<div class="cls-cards">
<div class="cls-card">Всего документов<strong>{label_total_documents:,}</strong></div>
<div class="cls-card">Заданных характеристик<strong>{label_metrics.height}</strong></div>
<div class="cls-card">Активных пар id–label<strong>{label_facts.height:,}</strong></div>
</div>
<p class="cls-note">Для каждой характеристики: доля = 100 × число уникальных документов с ней / {label_total_documents:,}.
Сумма долей может превышать 100%. Учитываются все 30 характеристик, включая отсутствующие.</p>
<div class="cls-columns">
<section class="cls-panel"><h2>Доля документов, % · число документов</h2>
<p class="cls-note">Единая шкала 0–100%; порядок по убыванию частоты.</p>{label_bar_rows}</section>
<section class="cls-panel"><h2>Метрики по всем характеристикам</h2>
<div class="cls-scroll"><table><thead><tr><th>Характеристика</th><th>Документов</th><th>Доля</th></tr></thead>
<tbody>{label_table_rows}</tbody></table></div></section>
</div>
<details><summary>Таблица фактов: первые 12 строк из {label_facts.height:,}</summary>
<p>Полная таблица доступна в label_facts; метрики — в label_metrics.</p>
<div class="cls-scroll"><table><thead><tr><th>id</th><th>label</th><th>value</th></tr></thead>
<tbody>{label_fact_rows}</tbody></table></div></details>
</article>
"""
label_report_html = (
    '<!doctype html><html lang="ru"><head><meta charset="utf-8">'
    '<meta name="viewport" content="width=device-width, initial-scale=1">'
    "<title>NEREL — EDA характеристик документов</title></head><body>"
    + label_report_fragment
    + "</body></html>"
)
label_report_path = Path("reports/nerel_task2.html")
label_report_path.parent.mkdir(parents=True, exist_ok=True)
label_report_path.write_text(label_report_html, encoding="utf-8")
display(HTML(label_report_fragment))
print(f"HTML-отчёт сохранён: {label_report_path.resolve()}")

Характеристика,Документов,Доля
ALTERNATIVE_NAME,723,96.92%
WORKS_AS,678,90.88%
PARTICIPANT_IN,656,87.94%
WORKPLACE,609,81.64%
POINT_IN_TIME,600,80.43%
TAKES_PLACE_IN,545,73.06%
AGE_IS,535,71.72%
HEADQUARTERED_IN,518,69.44%
LOCATED_IN,454,60.86%
ORIGINS_FROM,449,60.19%


HTML-отчёт сохранён: /Users/dima/Practikum/nlp-2026/sprint_1/src/project/reports/nerel_task2.html


# Выводы по EDA
1. Может быть документ в датасете со всеми нулями в метках документа.
1. Есть дисбаланс меток по частотам в документах. Топ15 меток встречаются почти в каждом документе. Серединка списка - в 50/50, 10 самых редких меток - у нескольких документов всего
1. по частоте меток на токен дисбаланс есть, но менее выражен
1. Самая редкая метка документа Penalized As, самая частая - alternative name
1. В среднем документы имеют длину до 256 токенов, максимальная длина - 1700

##### 3. Токенизация и выравнивание меток

Задачи:

- Выбрать `AutoTokenizer(..., use_fast=True)`.
- Реализовать `tokenize_and_align_labels(examples, tokenizer, label2id, max_length)`:
  - Токенизировать текст (return_offsets_mapping
  - Преобразовать word-level BIO метки в token-level метки (subword → label = -100 / ignore_index, для первых субтокенов ставится соответствующий тег `B-`/`I-`)
  - Вернуть словарь с `input_ids`, `attention_mask`, `labels` (token-level), `cls_labels`

- Собрать `torch.utils.data.Dataset` и `DataLoader`. Можно использовать `DataCollatorForTokenClassification` либо сделать кастомный collator, который возвращает батчи с `cls_labels`.



Task 3 build tokenization function

def tokenize(documents: list[BaseNerelExample], tokenizer: BaseTokenizer) -> list[BaseTokenizedNerelExample]
BaseNerelExample - корректный, правильный пример без нарушений целостности, проверенных ранее в EDA
BaseTokenizedNerelExample - TypedDict:
input_ids = list of list[int] - ids of subword tokens
attention_mask = list of list[int] - attention mask to subword tokens
labels = list of list[int] - labels of subword tokens, projected from NerelExample word tokens. 
Only first subword token gets word label, others get label = IGNORE_LABEL = -100


use this entities internally:

def project(word_labels: list[int], subword_word_ids: list[int]) -> subword labels
    returns list of len == len(subword_word_ids) with subword projected labels
Example:
word_labels = [54, 65, 2, 89, 321]
subword_word_ids = [0, 0, 1, 1, 1, 2, 3, 4, 4]
max(subword_word_ids) == len(word_labels) - subword_word_ids maps position to word_labels
output:
subword_labels = [54,  -100, 65, -100, -100, 2, 89, 321, -100]
explanation:
0, 0 -> 54,  -100: word_labels[0] == 54, -100 - second token of the same word

class Tokenizer - interface for pytorch tokenization:

def tokenize(self, documents: list[BaseNerelExample]) -> BaseTokenized
BaseNerelExample.tokens - input tokens for tokenization
truncation - нет. Эта токенизация будет использоваться как для трейна, так и для инференса. А на инференсе
нужно размечать все токены. 
padding="max_length"

BaseTokenized - interface to tokenized sequences, TypedDict:
input_ids = list of list[int]
attention_mask = list of list[int]

get_word_ids(batch_index: int) -> list[int] - return list of position numbers word tokens in NerelExample.tags

implement base entity for Tokenized, Tokenized and implemenatition with pytorch, datasets api


classes with Base* can not link to pytorch libraries, but you can implement child classes

Preprocess dataset ds with this tokenization function

collect results into entity
class BaseNerelDataset(list[BaseNerelExample]):
    input_ids: list[int]
    attentioin_mask: list[int]
    labels: list[int]
    cls_vec: list[int]

    def save(self, path) -> None

    @classmethod
    def load(self, path) -> Dataset


### Реализация Task 3

Базовые интерфейсы используют только стандартную библиотеку Python и заданные сущности `Base*`. `BaseTokenizer.tokenize()` принимает `BaseNerelExample` и возвращает `BaseTokenized`. Батч предоставляет только `input_ids`, `attention_mask` и `get_word_ids()`. `BaseTokenizedNerelExample` описывает один документ с полями `input_ids`, `attention_mask`, `labels` и `cls_vec`.

В условии смешаны формы одного документа и батча: `tokenize()` возвращает список документов, поэтому внутри `BaseTokenizedNerelExample` поля имеют тип `list[int]`; внутри `BaseTokenized` — `list[list[int]]`. `BaseNerelDataset` хранит список уже токенизированных `BaseTokenizedNerelExample`. Поля из эскиза датасета относятся к его записям. Помимо унаследованных операций списка интерфейс датасета предоставляет только `save(path)` и `load(path)`.

`HFTokenizer` и `HFTokenized` явно реализуют соответствующие базовые интерфейсы. Библиотечный токенизатор, настройки и таблица соответствия субтокенов словам скрыты в приватных полях. Остальные объекты обращаются к этим реализациям только через члены базовых интерфейсов. `NerelDataset` реализует сохранение и загрузку через Hugging Face; `Self` сохраняет тип наследника при загрузке.

`project` переносит BIO-метку только на первый субтокен исходного слова. Остальные субтокены, спецтокены и padding получают `IGNORE_LABEL = -100`. `None` в `word_ids` обозначает спецтокен или padding. Готовое разбиение `tokens` сохраняется с помощью `is_split_into_words=True`.

`max_length=512` задаёт длину дополнения, обрезки нет (`truncation=False`). Документ длиннее 512 субтокенов остаётся целым, поэтому длины строк могут различаться. Перед подачей в модель потребуется учесть её предел позиций и собрать тензоры подходящих размеров; это следующий этап.

Функция `preprocess_dataset(ds, tokenizer)` связывает Hugging Face Dataset с базовой функцией `tokenize()`: преобразует колонки батча в `BaseNerelExample` и собирает результат в `BaseNerelDataset`. Это отдельная функция, поскольку обработка исходного Hugging Face Dataset не входит в заданный интерфейс `BaseNerelDataset`. Внешние объекты не используют дополнительные методы его наследника. Сохранение маппингов вместе с данными реализуется в Task 4.

Справочники `label2id` и `id2label` строятся один раз по `tag_names` из EDA. При обучении их нужно сохранить вместе с моделью и повторно использовать на инференсе.


In [42]:
from abc import ABC, abstractmethod
from typing import Protocol, Self

IGNORE_LABEL = -100


class BaseTokenizedNerelExample(TypedDict):
    input_ids: list[int]
    attention_mask: list[int]
    labels: list[int]
    cls_vec: list[int]


class BaseNerelDataset(list[BaseTokenizedNerelExample], ABC):
    """Список документов и интерфейс хранения без библиотечных зависимостей."""

    @abstractmethod
    def save(self, path: str) -> None: ...

    @classmethod
    @abstractmethod
    def load(cls, path: str) -> Self: ...


class BaseTokenized(Protocol):
    input_ids: list[list[int]]
    attention_mask: list[list[int]]

    def get_word_ids(self, batch_index: int) -> list[int | None]: ...


class BaseTokenizer(Protocol):
    def tokenize(self, documents: list[BaseNerelExample]) -> BaseTokenized: ...


def project(word_labels: list[int], subword_word_ids: list[int | None]) -> list[int]:
    """Метка только у первого субтокена слова; спецтокены получают -100."""
    labels: list[int] = []
    previous_word_id: int | None = None
    for word_id in subword_word_ids:
        if word_id is None:
            labels.append(IGNORE_LABEL)
            continue
        labels.append(
            word_labels[word_id] if word_id != previous_word_id else IGNORE_LABEL
        )
        previous_word_id = word_id
    return labels

In [43]:
from dataclasses import dataclass

from transformers import AutoTokenizer, PreTrainedTokenizerFast


@dataclass
class HFTokenized(BaseTokenized):
    input_ids: list[list[int]]
    attention_mask: list[list[int]]
    _word_ids: list[list[int | None]]

    def get_word_ids(self, batch_index: int) -> list[int | None]:
        return self._word_ids[batch_index]


class HFTokenizer(BaseTokenizer):
    def __init__(self, backend: PreTrainedTokenizerFast, max_length: int = 512) -> None:
        self._backend = backend
        self._max_length = max_length

    def tokenize(self, documents: list[BaseNerelExample]) -> BaseTokenized:
        if not documents:
            return HFTokenized([], [], [])
        encoded = self._backend(
            [document["tokens"] for document in documents],
            is_split_into_words=True,
            truncation=False,
            padding="max_length",
            max_length=self._max_length,
            return_attention_mask=True,
            return_offsets_mapping=True,
            return_token_type_ids=False,
            return_tensors=None,
        )
        # Без return_tensors батч Hugging Face содержит списки Python.
        return HFTokenized(
            input_ids=cast(list[list[int]], encoded["input_ids"]),
            attention_mask=cast(list[list[int]], encoded["attention_mask"]),
            _word_ids=[encoded.word_ids(i) for i in range(len(documents))],
        )

`tokenize()` принимает `BaseTokenizer` и работает только с заданными членами `BaseTokenized`. `cls_vec` переносится без изменения порядка 30 характеристик.

По условию Task 3 все входные `BaseNerelExample` корректны: целостность уже проверена в EDA и повторно не проверяется. `tokenize`, `project` и адаптер `HFTokenizer` выполняют только преобразование данных. Fast-токенизатор выбирается через `use_fast=True`.


In [44]:
def tokenize(
    documents: list[BaseNerelExample], tokenizer: BaseTokenizer
) -> list[BaseTokenizedNerelExample]:
    """Токенизирует корректные примеры с целостностью, проверенной на этапе EDA."""
    if not documents:
        return []
    encoded = tokenizer.tokenize(documents)
    return [
        BaseTokenizedNerelExample(
            input_ids=encoded.input_ids[index],
            attention_mask=encoded.attention_mask[index],
            labels=project(
                [label2id[tag] for tag in document["tags"]],
                encoded.get_word_ids(index),
            ),
            cls_vec=document["cls_vec"].copy(),
        )
        for index, document in enumerate(documents)
    ]

In [46]:
from datasets import Dataset as HFDataset
from datasets import load_from_disk


class NerelBatch(TypedDict):
    tokens: list[list[str]]
    tags: list[list[str]]
    cls_vec: list[list[int]]


class NerelDataset(BaseNerelDataset):
    """Хранение и загрузка документов через Hugging Face Dataset."""

    def save(self, path: str) -> None:
        records = HFDataset.from_list([dict(record) for record in self])
        records.save_to_disk(path)

    @classmethod
    def load(cls, path: str) -> Self:
        records = cast(HFDataset, load_from_disk(path))
        return cls(cast(Iterable[BaseTokenizedNerelExample], records))


def preprocess_dataset(source: HFDataset, tokenizer: BaseTokenizer) -> BaseNerelDataset:
    """Преобразует библиотечные батчи в документы базового интерфейса."""

    def tokenize_batch(batch: NerelBatch) -> dict[str, list[list[int]]]:
        documents = [
            BaseNerelExample(tokens=tokens, tags=tags, cls_vec=cls_vec)
            for tokens, tags, cls_vec in zip(
                batch["tokens"], batch["tags"], batch["cls_vec"]
            )
        ]
        records = tokenize(documents, tokenizer)
        # Dataset.map(batched=True) принимает словарь колонок.
        return {
            "input_ids": [record["input_ids"] for record in records],
            "attention_mask": [record["attention_mask"] for record in records],
            "labels": [record["labels"] for record in records],
            "cls_vec": [record["cls_vec"] for record in records],
        }

    records = source.map(
        tokenize_batch,
        batched=True,
        batch_size=32,
        remove_columns=source.column_names,
        keep_in_memory=True,
        desc="Task 3: токенизация и выравнивание BIO-меток",
    )
    return NerelDataset(cast(Iterable[BaseTokenizedNerelExample], records))


TOKENIZER_NAME = "cointegrated/rubert-tiny2"
fast_tokenizer = cast(
    PreTrainedTokenizerFast,
    AutoTokenizer.from_pretrained(TOKENIZER_NAME, use_fast=True),
)
tokenizer: BaseTokenizer = HFTokenizer(fast_tokenizer, max_length=512)

tokenized_ds: BaseNerelDataset = preprocess_dataset(cast(HFDataset, ds), tokenizer)
print(f"Task 3: собрано документов в NerelDataset: {len(tokenized_ds)}")

Task 3: токенизация и выравнивание BIO-меток: 100%|██████████| 746/746 [00:00<00:00, 1631.00 examples/s]

Task 3: собрано документов в NerelDataset: 746


In [47]:
from tempfile import TemporaryDirectory

# Пример из условия, включая спецтокены и padding (word_id=None).
assert project([54, 65, 2, 89, 321], [None, 0, 0, 1, 1, 1, 2, 3, 4, 4, None]) == [
    IGNORE_LABEL,
    54,
    IGNORE_LABEL,
    65,
    IGNORE_LABEL,
    IGNORE_LABEL,
    2,
    89,
    321,
    IGNORE_LABEL,
    IGNORE_LABEL,
]
assert tokenize([], tokenizer) == []

assert len(tokenized_ds) == len(ds)
max_subword_length = 0
for source, tokenized_record in zip(
    cast(Iterable[BaseNerelExample], ds),
    cast(Iterable[BaseTokenizedNerelExample], tokenized_ds),
    strict=True,
):
    assert len(tokenized_record["input_ids"]) == len(tokenized_record["attention_mask"])
    assert len(tokenized_record["input_ids"]) == len(tokenized_record["labels"])
    # После удаления -100 должна восстановиться исходная последовательность меток.
    assert [label for label in tokenized_record["labels"] if label != IGNORE_LABEL] == [
        label2id[tag] for tag in source["tags"]
    ]
    assert tokenized_record["cls_vec"] == source["cls_vec"]
    assert all(
        label == IGNORE_LABEL
        for label, mask in zip(
            tokenized_record["labels"], tokenized_record["attention_mask"], strict=True
        )
        if mask == 0
    )
    max_subword_length = max(
        max_subword_length, sum(tokenized_record["attention_mask"])
    )

# Адаптер сохраняет и восстанавливает все записи, включая длинные документы.
with TemporaryDirectory(prefix="nerel-task3-") as temporary_path:
    tokenized_ds.save(temporary_path)
    restored_tokenized_ds = NerelDataset.load(temporary_path)
assert isinstance(restored_tokenized_ds, NerelDataset)
assert restored_tokenized_ds == tokenized_ds

print(f"Проверено и восстановлено документов: {len(tokenized_ds)}")
print(f"Максимальная длина без padding: {max_subword_length} субтокенов")
print(f"NER-меток: {len(label2id)}; CLS-характеристик: 30")

Saving the dataset (1/1 shards): 100%|██████████| 746/746 [00:00<00:00, 132034.38 examples/s]


Проверено и восстановлено документов: 746
Максимальная длина без padding: 2394 субтокенов
NER-меток: 58; CLS-характеристик: 30


Task 4 implement dataset storing to a local storage

class BaseNerelDatasetBundle:
    def label2id(self, label:str) -> int
    def id2label(self, id: int) -> str
    def dataset(self) -> BaseNerelDataset

class BaseDatasetStorage:
def load(self, path: str) -> BaseNerelDatasetBundle

def save(self, path: str, dataset: BaseNerelDatasetBundle) -> None

Ограничения:
для сохранения и загрузки BaseNerelDataset можно использовать только его собственные методы. 
Сущности с Base* могут взаимодействовать только с другими Base* сущностями.
Нельзя имплементировать новые сущности Base*, переменные и методы в ТЗ по сущности Base* - это 
те и только те члены класса, которые могут использовать другие объекты у любого наследника данного класса.

Проверки.
dataset = NerelDataset()

# save 
# load
loaded_dataset = ...

len(loaded_dataset.get_dataset()) == len(dataset.get_dataset())
len(loaded_dataset.get_dataset()) == len(dataset.get_dataset())
loaded_dataset.get_dataset()[0] == dataset.get_dataset()[0]
loaded_dataset.get_dataset()[7] == dataset.get_dataset()[7]

loaded_dataset.id2label(5) = dataset.id2label(5)
loaded_dataset.id2label(15) = dataset.id2label(15)

### Реализация Task 4: базовые интерфейсы и локальное хранилище

`BaseNerelDatasetBundle` содержит только `label2id`, `id2label` и `dataset`; `BaseDatasetStorage` — только `load` и `save`. Их абстрактные методы используют базовые интерфейсы, а реализации наследуют эти классы. Новых интерфейсов `Base*` и дополнительных членов в заданных интерфейсах нет.

`NerelDatasetBundle` хранит переданный `BaseNerelDataset` и маппинг NER-меток. `dataset()` возвращает этот же датасет без преобразования записей и без PyTorch-обёртки. Поля `input_ids`, `attention_mask`, `labels` и `cls_vec` сохраняют представление из Task 3.

`DatasetStorage` получает класс датасета с интерфейсом `BaseNerelDataset` и полный список имён NER-меток. Этот список нужен потому, что интерфейс bundle позволяет запросить id по имени, но не перечислить все метки. Он включает и метки, отсутствующие в сохраняемой выборке. Хранилище обращается к bundle только через `dataset()` и `label2id()`, а данные сохраняет и загружает только через `BaseNerelDataset.save()` и `BaseNerelDataset.load()`. Выбор Arrow остаётся внутри реализации `NerelDataset` из Task 3.

Локальная папка содержит `data/`, записанную самим датасетом, и `metadata.json` с маппингом `label_to_id`. После загрузки обратный маппинг восстанавливается из сохранённых id без сортировки или перенумерации. CLS-вектор хранится внутри записей датасета.


In [49]:
import json
from collections.abc import Mapping, Sequence
from pathlib import Path
from tempfile import TemporaryDirectory


class BaseNerelDatasetBundle(ABC):
    @abstractmethod
    def label2id(self, label: str) -> int: ...

    @abstractmethod
    def id2label(self, id: int) -> str: ...

    @abstractmethod
    def dataset(self) -> BaseNerelDataset: ...


class BaseDatasetStorage(ABC):
    @abstractmethod
    def load(self, path: str) -> BaseNerelDatasetBundle: ...

    @abstractmethod
    def save(self, path: str, dataset: BaseNerelDatasetBundle) -> None: ...


class NerelDatasetBundle(BaseNerelDatasetBundle):
    """Датасет Task 3 и маппинг NER-меток без изменения записей."""

    def __init__(
        self,
        records: BaseNerelDataset,
        label_to_id: Mapping[str, int],
    ) -> None:
        self._dataset = records
        self._label_to_id = dict(label_to_id)
        self._id_to_label = {index: label for label, index in label_to_id.items()}

    def label2id(self, label: str) -> int:
        return self._label_to_id[label]

    def id2label(self, id: int) -> str:
        return self._id_to_label[id]

    def dataset(self) -> BaseNerelDataset:
        return self._dataset


class DatasetStorage(BaseDatasetStorage):
    """Делегирует хранение записей датасету; отдельно сохраняет маппинг."""

    def __init__(
        self,
        dataset_type: type[BaseNerelDataset],
        label_names: Sequence[str],
    ) -> None:
        self._dataset_type = dataset_type
        self._label_names = tuple(label_names)

    def save(self, path: str, dataset: BaseNerelDatasetBundle) -> None:
        target = Path(path)
        if target.exists():
            raise FileExistsError(f"Путь уже существует: {target}; выберите новый путь")
        target.parent.mkdir(parents=True, exist_ok=True)
        # Публикуем папку только после успешной записи данных и маппинга.
        with TemporaryDirectory(prefix=f".{target.name}-", dir=target.parent) as tmp:
            staged = Path(tmp) / "bundle"
            staged.mkdir()
            dataset.dataset().save(str(staged / "data"))
            metadata = {
                "label_to_id": {
                    label: dataset.label2id(label) for label in self._label_names
                },
            }
            (staged / "metadata.json").write_text(
                json.dumps(metadata, ensure_ascii=False, indent=2) + "\n",
                encoding="utf-8",
            )
            staged.rename(target)

    def load(self, path: str) -> BaseNerelDatasetBundle:
        source = Path(path)
        metadata = cast(
            dict[str, dict[str, int]],
            json.loads((source / "metadata.json").read_text(encoding="utf-8")),
        )
        records = self._dataset_type.load(str(source / "data"))
        return NerelDatasetBundle(records, metadata["label_to_id"])

### Сохранение, загрузка и проверки из условия

В примере проверок условия `NerelDataset()` обозначает объект с датасетом и маппингами. Здесь эту роль выполняет `NerelDatasetBundle`, поскольку имя `NerelDataset` уже занято реализацией из Task 3. В примере также используется `get_dataset()`, однако заданный интерфейс объявляет только `dataset()`: проверки ниже используют именно этот метод. Повторяющаяся в условии проверка длины выполнена один раз.

Сохраняем результат Task 3 и загружаем bundle через `BaseDatasetStorage`. Проверки ограничены указанными в Task 4: длина датасета, записи с индексами 0 и 7, значения `id2label(5)` и `id2label(15)`. Записи состоят из списков Python, поэтому для сравнения достаточно `==`.

Запись сначала выполняется во временную папку рядом с целевой. Существующая папка не перезаписывается; при повторном запуске пример загружает её. После изменения данных или маппинга нужно выбрать новый путь. Для загрузки нужны определения классов и класс датасета, переданный хранилищу; исходный `ds` и токенизатор не нужны.


In [ ]:
nerel_dataset: BaseNerelDatasetBundle = NerelDatasetBundle(tokenized_ds, label2id)
storage: BaseDatasetStorage = DatasetStorage(NerelDataset, tag_names)
dataset_path = Path("artifacts/nerel_task4")

if not dataset_path.exists():
    storage.save(str(dataset_path), nerel_dataset)
restored_nerel_dataset = storage.load(str(dataset_path))

# Проверки из Task 4; dataset() — метод заданного базового интерфейса.
assert len(restored_nerel_dataset.dataset()) == len(nerel_dataset.dataset())
assert restored_nerel_dataset.dataset()[0] == nerel_dataset.dataset()[0]
assert restored_nerel_dataset.dataset()[7] == nerel_dataset.dataset()[7]
assert restored_nerel_dataset.id2label(5) == nerel_dataset.id2label(5)
assert restored_nerel_dataset.id2label(15) == nerel_dataset.id2label(15)

print(
    f"Task 4: сохранено и восстановлено документов: {len(restored_nerel_dataset.dataset())}"
)
print(f"Папка датасета: {dataset_path.resolve()}")

Task 4: сохранено и восстановлено документов: 746
Папка датасета: /Users/dima/Practikum/nlp-2026/sprint_1/src/project/artifacts/nerel_task4


Task 5 Model evaluation


class BaseBertModelInput:
input_ids - list[int]
attention_mask - list[int]

class BaseBertModelLabeldInput:
input_ids - list[int]
attention_mask - list[int]
labels - list[int]
document_labels list[int]

class BaseBertModelOutput:
text_classes: list[int]
token_tags: list[int]

class BaseModel:
    def infer(self, batch_input: list[BaseBertModelInput]) -> list[BaseBertModelOutput]

    def infer(self, batch_input: list[BaseBertModelLabeldInput]) -> list[BaseBertModelOutput]
    # everything from def infer(self, batch_input: list[BaseBertModelInput]) -> list[BaseBertModelOutput]
    # and filter out tags from output.token_tags, where labels[i] == IGNORE_LABEL at the same position

    def save(self, path)

    def load(self, path)

Implement specific BaseModel - random model, which returns random choice of token tags, random labels from document labels


class BaseMetric:
    def eval(self, y_true: list[BaseBertModelLabeldInput], y_pred: list[BaseBertModelOutput]) -> float

    def get_name(self) -> str


class BaseMetricEvaluator:
# storing and aggregating metrics
def add(self, metric: BaseMetric) -> None

def eval(self, y_true: list[BaseBertModelLabeldInput], y_pred: list[BaseBertModelOutput]) -> dict[str, float]


Implement two sets of metrics:
1) token prediction metrics:
implement f1-macro-tokens, precision-tokens, recall-tokens metrics in BaseMetric. Concern token fields in both true and pred. Use sklearn implementations
implement specific BaseMetricEvaluator 

2) document labels prediction
implement f1-macro-topics, precision-topics, recall-topics metrics in BaseMetric. Concern document topics fields in both true and pred. Use sklearn implementations
implement specific BaseMetricEvaluator
Every topic label in document - separate binary target, not collectively exhastuve labels


eval random model against this set of metrics and print report

### Реализация Task 5: случайная модель и оценка NER + CLS

Базовые интерфейсы содержат только члены из условия и используют стандартную библиотеку Python. Входы и выход описаны через `TypedDict`, как записи предыдущих заданий. Написание `BaseBertModelLabeldInput` сохранено из условия. Его `document_labels` заполняется исходным `cls_vec`: это multihot-вектор из 30 значений 0/1 в порядке `CLS_LABELS`. Выходной `text_classes` имеет тот же формат и порядок.

`RandomModel` равномерно выбирает BIO-теги из полного сохранённого маппинга, включая `O`. Для каждой характеристики документа независимо выбирается 0 или 1 с вероятностью 0,5. Возможны несколько единиц одновременно или полностью нулевой вектор. Истинные BIO-метки и характеристики не используются для случайного выбора. Две перегрузки `infer()` поддерживают обычные и размеченные батчи: в размеченном случае после генерации удаляются позиции с `labels == IGNORE_LABEL`.

Используется собственный генератор с seed 42. `save()` записывает конфигурацию и состояние генератора в JSON-файл; `load()` восстанавливает их в текущий экземпляр, чтобы продолжить последовательность случайных предсказаний.

Все шесть метрик используют `sklearn.metrics`: `f1_score`, `precision_score`, `recall_score`, с `average="macro"` и `zero_division=0`. F1 усредняет F1 отдельных меток. Три реализации `BaseMetric` параметризованы областью оценки (`tokens` или `topics`) и регистрируются в двух независимых экземплярах `MetricEvaluator`.

Для NER объединяем документы в последовательность BIO-тегов первых субтокенов слов. Истинные `IGNORE_LABEL` исключаются внутри метрик, предсказания уже отфильтрованы моделью. Усреднение идёт по полному набору NER-меток, включая `O`. Это оценка классификации токенов, а не совпадений сущностей целиком.

Для CLS передаём sklearn две матрицы формы «документы × 30 характеристик»: `document_labels` и `text_classes`. Каждая колонка — независимая бинарная цель; macro-усреднение выполняется по всем 30 характеристикам. `argmax` и преобразование в единственный класс не используются.

`FixedTopicsRandomModel` наследует `RandomModel` и заменяет CLS-выход для каждого документа: `ALTERNATIVE_NAME`, `WORKS_AS`, `PARTICIPANT_IN`, `WORKPLACE` всегда равны 1, остальные характеристики — 0. Позиции определяются по именам в `CLS_LABELS` и сохранённым индексам характеристик. Для каждого документа создаётся отдельный список. BIO-предсказания, фильтрация `IGNORE_LABEL`, сохранение и загрузка наследуются. При одинаковом seed BIO-предсказания обеих моделей совпадают: наследник сначала вызывает исходный `infer()`, затем заменяет только `text_classes`. Для загрузки этой версии нужно создать `FixedTopicsRandomModel` и вызвать у него `load()`.


В дополнительном эксперименте `selected_topic_evaluator` оценивает только `ALTERNATIVE_NAME`, `WORKS_AS`, `PARTICIPANT_IN`, `WORKPLACE`. Их исходные индексы передаются в `labels` функций sklearn: из macro-усреднения исключаются остальные 26 колонок, а все документы и предсказания сохраняются. В отчёте сопоставлены результаты обеих моделей по 30 и по 4 характеристикам, а также показана фактическая встречаемость выбранных меток.

Если всегда предсказывать выбранную характеристику, её recall равен 1, precision равен доле положительных документов `p`, а F1 равен `2p/(1+p)`. Поэтому отбор частых характеристик позволяет получить высокий macro F1 без анализа текста. При оценке по всем 30 характеристикам оставшиеся 26 дают нулевой F1 и снижают среднее. Отчёт по четырём меткам описывает только выбранную подзадачу; он не оценивает качество по полному набору характеристик.


In [ ]:
from typing import overload


class BaseBertModelInput(TypedDict):
    input_ids: list[int]
    attention_mask: list[int]


class BaseBertModelLabeldInput(BaseBertModelInput):
    labels: list[int]
    document_labels: list[int]


class BaseBertModelOutput(TypedDict):
    text_classes: list[int]
    token_tags: list[int]


class BaseModel(ABC):
    @overload
    def infer(
        self, batch_input: list[BaseBertModelInput]
    ) -> list[BaseBertModelOutput]: ...

    @overload
    def infer(
        self, batch_input: list[BaseBertModelLabeldInput]
    ) -> list[BaseBertModelOutput]: ...

    @abstractmethod
    def infer(
        self,
        batch_input: list[BaseBertModelInput] | list[BaseBertModelLabeldInput],
    ) -> list[BaseBertModelOutput]: ...

    @abstractmethod
    def save(self, path: str) -> None: ...

    @abstractmethod
    def load(self, path: str) -> None: ...


class BaseMetric(ABC):
    @abstractmethod
    def eval(
        self,
        y_true: list[BaseBertModelLabeldInput],
        y_pred: list[BaseBertModelOutput],
    ) -> float: ...

    @abstractmethod
    def get_name(self) -> str: ...


class BaseMetricEvaluator(ABC):
    @abstractmethod
    def add(self, metric: BaseMetric) -> None: ...

    @abstractmethod
    def eval(
        self,
        y_true: list[BaseBertModelLabeldInput],
        y_pred: list[BaseBertModelOutput],
    ) -> dict[str, float]: ...

In [ ]:
from random import Random


class RandomModel(BaseModel):
    """Равномерный случайный baseline с сохраняемым состоянием генератора."""

    def __init__(
        self, tag_ids: Sequence[int], class_ids: Sequence[int], seed: int = 42
    ) -> None:
        self._tag_ids = list(tag_ids)
        self._class_ids = list(class_ids)
        self._random = Random(seed)

    @overload
    def infer(
        self, batch_input: list[BaseBertModelInput]
    ) -> list[BaseBertModelOutput]: ...

    @overload
    def infer(
        self, batch_input: list[BaseBertModelLabeldInput]
    ) -> list[BaseBertModelOutput]: ...

    def infer(
        self,
        batch_input: list[BaseBertModelInput] | list[BaseBertModelLabeldInput],
    ) -> list[BaseBertModelOutput]:
        predictions: list[BaseBertModelOutput] = []
        for document in batch_input:
            token_tags = [
                self._random.choice(self._tag_ids) for _ in document["input_ids"]
            ]
            text_classes = [self._random.choice((0, 1)) for _ in self._class_ids]
            if "labels" in document:
                labels = cast(BaseBertModelLabeldInput, document)["labels"]
                token_tags = [
                    tag
                    for tag, label in zip(token_tags, labels)
                    if label != IGNORE_LABEL
                ]
            predictions.append(
                BaseBertModelOutput(text_classes=text_classes, token_tags=token_tags)
            )
        return predictions

    def save(self, path: str) -> None:
        target = Path(path)
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_text(
            json.dumps(
                {
                    "tag_ids": self._tag_ids,
                    "class_ids": self._class_ids,
                    "random_state": self._random.getstate(),
                },
                ensure_ascii=False,
                indent=2,
            )
            + "\n",
            encoding="utf-8",
        )

    def load(self, path: str) -> None:
        payload = json.loads(Path(path).read_text(encoding="utf-8"))
        self._tag_ids = cast(list[int], payload["tag_ids"])
        self._class_ids = cast(list[int], payload["class_ids"])
        version, internal_state, gauss_next = cast(
            tuple[int, list[int], float | None], payload["random_state"]
        )
        self._random.setstate((version, tuple(internal_state), gauss_next))


class FixedTopicsRandomModel(RandomModel):
    """Случайные BIO-теги и четыре постоянные положительные CLS-метки."""

    @overload
    def infer(
        self, batch_input: list[BaseBertModelInput]
    ) -> list[BaseBertModelOutput]: ...

    @overload
    def infer(
        self, batch_input: list[BaseBertModelLabeldInput]
    ) -> list[BaseBertModelOutput]: ...

    def infer(
        self,
        batch_input: list[BaseBertModelInput] | list[BaseBertModelLabeldInput],
    ) -> list[BaseBertModelOutput]:
        # Сохраняем BIO-предсказания и фильтрацию IGNORE_LABEL родителя.
        predictions = super().infer(batch_input)
        positive_topics = {
            "ALTERNATIVE_NAME",
            "WORKS_AS",
            "PARTICIPANT_IN",
            "WORKPLACE",
        }
        fixed_topics = [
            int(CLS_LABELS[class_id] in positive_topics) for class_id in self._class_ids
        ]
        for prediction in predictions:
            prediction["text_classes"] = fixed_topics.copy()
        return predictions

In [ ]:
from collections.abc import Callable
from typing import Literal

from sklearn.metrics import f1_score, precision_score, recall_score


def metric_targets(
    y_true: list[BaseBertModelLabeldInput],
    y_pred: list[BaseBertModelOutput],
    target: Literal["tokens", "topics"],
) -> tuple[list[int] | list[list[int]], list[int] | list[list[int]]]:
    """BIO-последовательности или multihot-матрицы характеристик документа."""
    if target == "tokens":
        return (
            [
                label
                for document in y_true
                for label in document["labels"]
                if label != IGNORE_LABEL
            ],
            [tag for prediction in y_pred for tag in prediction["token_tags"]],
        )
    return (
        [document["document_labels"] for document in y_true],
        [prediction["text_classes"] for prediction in y_pred],
    )


def sklearn_macro_score(
    scorer: Callable[..., float],
    actual: list[int] | list[list[int]],
    predicted: list[int] | list[list[int]],
    label_ids: Sequence[int],
) -> float:
    # sklearn не аннотирует эти функции: default="warn" сужает выводимый
    # Pyright тип zero_division до str, хотя API также принимает 0 и 1.
    return float(
        scorer(
            actual,
            predicted,
            labels=list(label_ids),
            average="macro",
            zero_division=0,
        )
    )


class F1MacroMetric(BaseMetric):
    def __init__(
        self, label_ids: Sequence[int], target: Literal["tokens", "topics"]
    ) -> None:
        self._label_ids = list(label_ids)
        self._target: Literal["tokens", "topics"] = target

    def eval(
        self,
        y_true: list[BaseBertModelLabeldInput],
        y_pred: list[BaseBertModelOutput],
    ) -> float:
        actual, predicted = metric_targets(y_true, y_pred, self._target)
        return sklearn_macro_score(f1_score, actual, predicted, self._label_ids)

    def get_name(self) -> str:
        return f"f1-macro-{self._target}"


class PrecisionMetric(BaseMetric):
    def __init__(
        self, label_ids: Sequence[int], target: Literal["tokens", "topics"]
    ) -> None:
        self._label_ids = list(label_ids)
        self._target: Literal["tokens", "topics"] = target

    def eval(
        self,
        y_true: list[BaseBertModelLabeldInput],
        y_pred: list[BaseBertModelOutput],
    ) -> float:
        actual, predicted = metric_targets(y_true, y_pred, self._target)
        return sklearn_macro_score(precision_score, actual, predicted, self._label_ids)

    def get_name(self) -> str:
        return f"precision-{self._target}"


class RecallMetric(BaseMetric):
    def __init__(
        self, label_ids: Sequence[int], target: Literal["tokens", "topics"]
    ) -> None:
        self._label_ids = list(label_ids)
        self._target: Literal["tokens", "topics"] = target

    def eval(
        self,
        y_true: list[BaseBertModelLabeldInput],
        y_pred: list[BaseBertModelOutput],
    ) -> float:
        actual, predicted = metric_targets(y_true, y_pred, self._target)
        return sklearn_macro_score(recall_score, actual, predicted, self._label_ids)

    def get_name(self) -> str:
        return f"recall-{self._target}"


class MetricEvaluator(BaseMetricEvaluator):
    def __init__(self) -> None:
        self._metrics: list[BaseMetric] = []

    def add(self, metric: BaseMetric) -> None:
        self._metrics.append(metric)

    def eval(
        self,
        y_true: list[BaseBertModelLabeldInput],
        y_pred: list[BaseBertModelOutput],
    ) -> dict[str, float]:
        return {
            metric.get_name(): metric.eval(y_true, y_pred) for metric in self._metrics
        }

In [ ]:
# Используем документы и сохранённый маппинг из Task 4.
evaluation_tag_ids = [restored_nerel_dataset.label2id(tag) for tag in tag_names]
evaluation_class_ids = list(range(len(CLS_LABELS)))
evaluation_inputs = [
    BaseBertModelLabeldInput(
        input_ids=record["input_ids"],
        attention_mask=record["attention_mask"],
        labels=record["labels"],
        document_labels=record["cls_vec"],
    )
    for record in restored_nerel_dataset.dataset()
]

random_model: BaseModel = RandomModel(evaluation_tag_ids, evaluation_class_ids, seed=42)
random_model_path = Path("artifacts/nerel_task5/random_model.json")
random_model.save(str(random_model_path))
# load() восстанавливает конфигурацию и генератор в текущем экземпляре.
random_model.load(str(random_model_path))

token_evaluator: BaseMetricEvaluator = MetricEvaluator()
token_evaluator.add(F1MacroMetric(evaluation_tag_ids, target="tokens"))
token_evaluator.add(PrecisionMetric(evaluation_tag_ids, target="tokens"))
token_evaluator.add(RecallMetric(evaluation_tag_ids, target="tokens"))

topic_evaluator: BaseMetricEvaluator = MetricEvaluator()
topic_evaluator.add(F1MacroMetric(evaluation_class_ids, target="topics"))
topic_evaluator.add(PrecisionMetric(evaluation_class_ids, target="topics"))
topic_evaluator.add(RecallMetric(evaluation_class_ids, target="topics"))

# BIO-фильтрация выполняется моделью; document_labels нужны только метрикам.
random_predictions = random_model.infer(evaluation_inputs)
random_token_metrics = token_evaluator.eval(evaluation_inputs, random_predictions)
random_topic_metrics = topic_evaluator.eval(evaluation_inputs, random_predictions)
evaluated_tokens = sum(len(item["token_tags"]) for item in random_predictions)

print("Task 5 — RandomModel: NER + CLS, macro, seed=42")
print(f"Документов: {len(evaluation_inputs):,}")
print(f"Оцениваемых токенов: {evaluated_tokens:,}")
print(f"BIO-меток (включая O): {len(evaluation_tag_ids)}")
print(f"Независимых CLS-характеристик: {len(evaluation_class_ids)}")
print("IGNORE_LABEL исключён; zero_division=0 для всех метрик.")
print("\nNER — классификация BIO-тегов токенов:")
for metric_name, metric_value in random_token_metrics.items():
    print(f"{metric_name:20s} {metric_value:.6f}")
print("\nCLS — независимые бинарные характеристики, случайный выбор 0/1 с p=0.5:")
for metric_name, metric_value in random_topic_metrics.items():
    print(f"{metric_name:20s} {metric_value:.6f}")
print(f"\nМодель: {random_model_path.resolve()}")

fixed_topics_model: BaseModel = FixedTopicsRandomModel(
    evaluation_tag_ids, evaluation_class_ids, seed=42
)
fixed_topics_model_path = Path("artifacts/nerel_task5/fixed_topics_random_model.json")
fixed_topics_model.save(str(fixed_topics_model_path))
fixed_topics_model.load(str(fixed_topics_model_path))
fixed_topics_predictions = fixed_topics_model.infer(evaluation_inputs)
fixed_topics_token_metrics = token_evaluator.eval(
    evaluation_inputs, fixed_topics_predictions
)
fixed_topics_topic_metrics = topic_evaluator.eval(
    evaluation_inputs, fixed_topics_predictions
)

print("\nTask 5 — FixedTopicsRandomModel: NER + CLS, macro, seed=42")
print("CLS = 1: ALTERNATIVE_NAME, WORKS_AS, PARTICIPANT_IN, WORKPLACE; остальные = 0.")
print("\nNER — классификация BIO-тегов токенов:")
for metric_name, metric_value in fixed_topics_token_metrics.items():
    print(f"{metric_name:20s} {metric_value:.6f}")
print("\nCLS — постоянный multihot-вектор:")
for metric_name, metric_value in fixed_topics_topic_metrics.items():
    print(f"{metric_name:20s} {metric_value:.6f}")
print(f"\nМодель: {fixed_topics_model_path.resolve()}")

# Эксперимент: меняем только набор колонок, участвующих в CLS-усреднении.
selected_topic_names = [
    "ALTERNATIVE_NAME",
    "WORKS_AS",
    "PARTICIPANT_IN",
    "WORKPLACE",
]
selected_topic_ids = [CLS_LABELS.index(name) for name in selected_topic_names]
selected_topic_evaluator: BaseMetricEvaluator = MetricEvaluator()
selected_topic_evaluator.add(F1MacroMetric(selected_topic_ids, target="topics"))
selected_topic_evaluator.add(PrecisionMetric(selected_topic_ids, target="topics"))
selected_topic_evaluator.add(RecallMetric(selected_topic_ids, target="topics"))
random_selected_topic_metrics = selected_topic_evaluator.eval(
    evaluation_inputs, random_predictions
)
fixed_selected_topic_metrics = selected_topic_evaluator.eval(
    evaluation_inputs, fixed_topics_predictions
)

print("\nЭксперимент: CLS-оценка только по четырём выбранным характеристикам")
print("Остальные 26 характеристик исключены из усреднения; все документы сохранены.")
print("\nДоля документов с каждой выбранной характеристикой:")
for topic_name, topic_id in zip(selected_topic_names, selected_topic_ids):
    topic_share = sum(
        document["document_labels"][topic_id] for document in evaluation_inputs
    ) / len(evaluation_inputs)
    print(f"{topic_name:20s} {topic_share:.2%}")

print("\nМодель                  Топиков  F1 macro  Precision  Recall")
for model_name, topic_count, scores in (
    ("RandomModel", len(evaluation_class_ids), random_topic_metrics),
    ("RandomModel", len(selected_topic_ids), random_selected_topic_metrics),
    ("FixedTopicsRandomModel", len(evaluation_class_ids), fixed_topics_topic_metrics),
    ("FixedTopicsRandomModel", len(selected_topic_ids), fixed_selected_topic_metrics),
):
    print(
        f"{model_name:23s} {topic_count:7d}"
        f"  {scores['f1-macro-topics']:.6f}"
        f"  {scores['precision-topics']:.6f}"
        f"  {scores['recall-topics']:.6f}"
    )
print(
    "\nВысокая оценка постоянного предсказания достигается выбором частых меток: "
    "ошибки по остальным 26 характеристикам больше не учитываются."
)

Task 5 — RandomModel: NER + CLS, macro, seed=42
Документов: 746
Оцениваемых токенов: 165,084
BIO-меток (включая O): 58
Независимых CLS-характеристик: 30
IGNORE_LABEL исключён; zero_division=0 для всех метрик.

NER — классификация BIO-тегов токенов:
f1-macro-tokens      0.007674
precision-tokens     0.017489
recall-tokens        0.017105

CLS — независимые бинарные характеристики, случайный выбор 0/1 с p=0.5:
f1-macro-topics      0.389790
precision-topics     0.398882
recall-topics        0.491347

Модель: /Users/dima/Practikum/nlp-2026/sprint_1/src/project/artifacts/nerel_task5/random_model.json

Task 5 — FixedTopicsRandomModel: NER + CLS, macro, seed=42
CLS = 1: ALTERNATIVE_NAME, WORKS_AS, PARTICIPANT_IN, WORKPLACE; остальные = 0.

NER — классификация BIO-тегов токенов:
f1-macro-tokens      0.007674
precision-tokens     0.017489
recall-tokens        0.017105

CLS — постоянный multihot-вектор:
f1-macro-topics      0.125710
precision-topics     0.119124
recall-topics        0.133333

Мо

# Выводы
По итогу этого блока я достроил пайплайн инференса модели с метриками, EDA, но без обучения нейросетевой модели
Подтвердил на практике одно из наблюдений EDA и проверил адекватность пайплайна в целом. Увидел, что если сделать модель, которая для топиков документа
предиктит топ4 самых частых топика и если оцениваться только по этим топикам, то можно выбить F1 94%. Делаю вывод, что код для EDA, препроцессинга и инференса
работоспособен, адекватен, дружит с здравым смыслом.

Далее буду отдельно обучать нейросетевую модель и встраивать ее в пайплайн оценки.

# Task 6 DataLoader

class TTorchModelInput:
input_ids - torch.Tensor
attention_mask - torch.Tensor
def to(self, device: torch.device) - move all tensors to device
def to_dict(self) -> TypedDict["input_ids", "attention_mask"]



class TTorchBatchCollator:
def collate(self, features: BaseBertModelInput) -> TTorchModelInput

class TTorchDataLoader - implements Iterator protocol:
def __iter__(self) -> TTorchModelInput

set_collator(self, TTorchBatchCollator) -> None


implement TTorchBatchCollator mirorring DataCollatorForTokenClassification and build several batches from BaseNerelDataset


implement TTorchDataLoader mirorring torch.utils.data from DataLoader and generate several batches from BaseNerelDataset at batch_size=6



### Реализация Task 6: collator и тензорные батчи

`TTorchModelInput` хранит два тензора формы `[batch_size, sequence_length]`. `to(device)` переносит оба тензора и возвращает тот же объект; `to_dict()` возвращает словарь для передачи аргументов модели.

`collate()` принимает последовательность `BaseBertModelInput`, поскольку один батч собирается из нескольких документов. Padding делегирован `DataCollatorForTokenClassification`: используются pad-токен и сторона padding исходного токенизатора, новые позиции `attention_mask` равны нулю, тензоры имеют тип `torch.int64`. Уже существующий padding сохраняется; обрезание последовательностей здесь не выполняется.

В заданном интерфейсе батча есть только `input_ids` и `attention_mask`. `labels` и `cls_vec` остаются в исходном датасете; для передачи целей обучения потребуется расширить интерфейс батча.

`TTorchDataLoader` делегирует батчинг `torch.utils.data.DataLoader` и реализует один проход по протоколу `Iterator`: `__iter__()` возвращает сам итератор, `__next__()` — следующий батч, после конца возникает `StopIteration`. Новый экземпляр или `set_collator()` начинает новый проход. Последний неполный батч сохраняется.


In [ ]:
from collections.abc import Iterator, Sequence
from dataclasses import dataclass
from typing import Self, TypedDict, cast

import torch
from torch.utils.data import DataLoader, Dataset
from transformers import DataCollatorForTokenClassification, PreTrainedTokenizerBase


class ModelInputTensors(TypedDict):
    input_ids: torch.Tensor
    attention_mask: torch.Tensor


@dataclass
class TTorchModelInput:
    input_ids: torch.Tensor
    attention_mask: torch.Tensor

    def to(self, device: torch.device) -> Self:
        self.input_ids = self.input_ids.to(device)
        self.attention_mask = self.attention_mask.to(device)
        return self

    def to_dict(self) -> ModelInputTensors:
        return ModelInputTensors(
            input_ids=self.input_ids,
            attention_mask=self.attention_mask,
        )


class TTorchBatchCollator:
    def __init__(self, tokenizer: PreTrainedTokenizerBase) -> None:
        self._collator = DataCollatorForTokenClassification(
            tokenizer=tokenizer,
            padding=True,
            return_tensors="pt",
        )

    def collate(self, features: Sequence[BaseBertModelInput]) -> TTorchModelInput:
        # Передаём только заданные поля: cls_vec и labels остаются в датасете.
        batch = self._collator(
            [
                {
                    "input_ids": feature["input_ids"],
                    "attention_mask": feature["attention_mask"],
                }
                for feature in features
            ]
        )
        return TTorchModelInput(
            input_ids=cast(torch.Tensor, batch["input_ids"]),
            attention_mask=cast(torch.Tensor, batch["attention_mask"]),
        )


class _NerelTorchDataset(Dataset[BaseBertModelInput]):
    """Адаптер BaseNerelDataset к интерфейсу Dataset в PyTorch."""

    def __init__(self, dataset: BaseNerelDataset) -> None:
        self._dataset = dataset

    def __len__(self) -> int:
        return len(self._dataset)

    def __getitem__(self, index: int) -> BaseBertModelInput:
        record = self._dataset[index]
        return BaseBertModelInput(
            input_ids=record["input_ids"],
            attention_mask=record["attention_mask"],
        )


class TTorchDataLoader(Iterator[TTorchModelInput]):
    """Один проход по датасету с батчингом через torch DataLoader."""

    def __init__(
        self,
        dataset: BaseNerelDataset,
        collator: TTorchBatchCollator,
        batch_size: int = 6,
        shuffle: bool = False,
    ) -> None:
        self._dataset = _NerelTorchDataset(dataset)
        self._batch_size = batch_size
        self._shuffle = shuffle
        self._iterator: Iterator[TTorchModelInput] | None = None
        self.set_collator(collator)

    def set_collator(self, collator: TTorchBatchCollator) -> None:
        # Новый collator начинает новый проход; неполный последний батч сохраняем.
        self._loader = DataLoader(
            self._dataset,
            batch_size=self._batch_size,
            shuffle=self._shuffle,
            collate_fn=collator.collate,
            drop_last=False,
            num_workers=0,
        )
        self._iterator = None

    def __iter__(self) -> Self:
        return self

    def __next__(self) -> TTorchModelInput:
        if self._iterator is None:
            self._iterator = iter(self._loader)
        return next(self._iterator)


In [ ]:
from itertools import islice

# Сохранённый BaseNerelDataset из Task 4, без копирования всех документов.
torch_source = restored_nerel_dataset.dataset()
torch_collator = TTorchBatchCollator(fast_tokenizer)
torch_device = torch.device("cpu")

# Несколько батчей напрямую через collate: до шести документов в каждом.
for start in range(0, min(18, len(torch_source)), 6):
    features = [
        BaseBertModelInput(
            input_ids=record["input_ids"],
            attention_mask=record["attention_mask"],
        )
        for record in torch_source[start : start + 6]
    ]
    collated_batch = torch_collator.collate(features).to(torch_device)
    print(
        f"Collator, батч {start // 6 + 1}: "
        f"input_ids={tuple(collated_batch.input_ids.shape)}, "
        f"attention_mask={tuple(collated_batch.attention_mask.shape)}"
    )

# Тот же батчинг через torch DataLoader и протокол Iterator.
torch_loader = TTorchDataLoader(torch_source, torch_collator, batch_size=6)
torch_loader.set_collator(torch_collator)
torch_batches = list(islice(torch_loader, 3))
for batch_number, torch_batch in enumerate(torch_batches, start=1):
    model_inputs = torch_batch.to(torch_device).to_dict()
    print(
        f"DataLoader, батч {batch_number}: "
        f"input_ids={tuple(model_inputs['input_ids'].shape)}, "
        f"attention_mask={tuple(model_inputs['attention_mask'].shape)}, "
        f"dtype={model_inputs['input_ids'].dtype}, "
        f"device={model_inputs['input_ids'].device}"
    )


Task 7 torch модель

class TTorchBackwardableModule(torch.nn.Module):
def backward(self) -> none



class TTorchModelMultitaskOutput:
loss : TTorchBackwardableModule
outputs: torch.Tensor - не понимаю, как должен выглядеть выходной тензор для мультитаскной модели

class TTrainableModel(BaseModel):
def train(self) -> None - switch model to train mode

def eval(self) -> switch model to eval mode

def to(self, device: torch.device) -> None - move model to torch device

def parameters(self) -> Iterator[torch.Parameter] - get torch like parameters

def torch_model(self) -> nn.Module - get compatible torch model for export

def train_predict(self, input: TTorchModelInput) -> TTorchModelOutput


Implement One Layer FullyConnected NN

##### 4. Модель: `JointModel` + custom loss (uncertainty weighting)


In [ ]:
from torch import nn


class JointModel(nn.Module):
    pass

##### 5. Training / Validation



##### 6. Инференс, квантизация и анализ ошибок

Проведите качественный анализ на 8–10 примерах: где NER ошибается? Какие типы сущностей плохо определяются? Насколько квантизация может ускорить инференс и сильно ли она ухудшает модель?


##### Заключение

Этот шаблон даёт вам чёткую дорожную карту и рабочие точки, где нужно реализовать код. В ноутбуке необходимо самостоятельно реализовать токенизацию/выравнивание меток, датасеты, модель, loss, тренировку и анализ.

Удачи - приступайте к реализации прямо в ноутбуке!